# 💳 Credit Card Fraud Detection — Exploratory Data Analysis
## American Express Style | Portfolio Project

---

### 📋 Table of Contents
1. [Setup & Configuration](#1)
2. [Data Loading & First Look](#2)
3. [Data Quality Assessment](#3)
4. [Class Imbalance Analysis](#4)
5. [Transaction Amount Analysis](#5)
6. [Time-Based Analysis](#6)
7. [PCA Feature Analysis (V1–V28)](#7)
8. [Correlation Analysis](#8)
9. [Fraud vs Legitimate — Feature Distributions](#9)
10. [Outlier & Anomaly Analysis](#10)
11. [Bivariate Analysis](#11)
12. [Key Insights Summary](#12)

---
> **Dataset:** 284,807 real credit card transactions (Sept 2013, European cardholders)  
> **Features:** 28 PCA-anonymized features (V1–V28) + Time + Amount + Class  
> **Goal:** Detect fraudulent transactions (Class=1) from legitimate ones (Class=0)  
> **Challenge:** Extreme class imbalance — only **0.17%** are fraudulent


## 1. Setup & Configuration <a id='1'></a>

In [ ]:
# ── Install dependencies if needed ────────────────────────────────────────
# !pip install pandas numpy matplotlib seaborn scipy

import warnings
warnings.filterwarnings('ignore')

# Core
import pandas as pd
import numpy as np

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ks_2samp

# Display settings
pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 120)
np.set_printoptions(suppress=True, precision=4)

# ── AmEx Color Palette ─────────────────────────────────────────────────────
C = {
    'blue'    : '#006FCF',   # AmEx Blue
    'red'     : '#E63946',   # Fraud Red
    'green'   : '#2DC653',   # Safe Green
    'dark'    : '#1A1A2E',   # Background
    'mid'     : '#16213E',   # Panel background
    'accent'  : '#0F3460',   # Borders
    'gold'    : '#F4A261',   # Highlights
    'white'   : '#FFFFFF',
    'gray'    : '#8892A4',
    'purple'  : '#7B2D8B',
    'teal'    : '#2EC4B6',
}

# ── Global plot style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : C['dark'],
    'axes.facecolor'   : C['mid'],
    'axes.edgecolor'   : C['accent'],
    'text.color'       : C['white'],
    'axes.labelcolor'  : C['white'],
    'xtick.color'      : C['white'],
    'ytick.color'      : C['white'],
    'grid.color'       : C['accent'],
    'grid.alpha'       : 0.35,
    'font.family'      : 'DejaVu Sans',
    'axes.titlepad'    : 12,
    'figure.dpi'       : 110,
})

FRAUD_COLOR = C['red']
LEGIT_COLOR = C['blue']

print("✅ Setup complete — AmEx theme loaded")
print(f"   Pandas  : {pd.__version__}")
print(f"   NumPy   : {np.__version__}")
print(f"   Seaborn : {sns.__version__}")


## 2. Data Loading & First Look <a id='2'></a>

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
df = pd.read_csv('data/creditcard.csv')

print("=" * 60)
print("  DATASET OVERVIEW")
print("=" * 60)
print(f"  Rows        : {df.shape[0]:,}")
print(f"  Columns     : {df.shape[1]}")
print(f"  Memory      : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Fraud cases : {df['Class'].sum():,}  ({df['Class'].mean()*100:.4f}%)")
print(f"  Legit cases : {(df['Class']==0).sum():,}  ({(df['Class']==0).mean()*100:.4f}%)")
print("=" * 60)


In [ ]:
# ── First 5 rows ──────────────────────────────────────────────────────────
print("First 5 rows:")
df.head()


In [ ]:
# ── Last 5 rows ───────────────────────────────────────────────────────────
print("Last 5 rows:")
df.tail()


In [ ]:
# ── Data types and non-null counts ────────────────────────────────────────
print("Column info:")
df.info(verbose=True, show_counts=True)


In [ ]:
# ── Statistical summary ───────────────────────────────────────────────────
print("Statistical Summary — All Features:")
df.describe().T.style.background_gradient(cmap='Blues', axis=0)


## 3. Data Quality Assessment <a id='3'></a>

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = df.isnull().mean() * 100

quality_df = pd.DataFrame({
    'Missing Count'  : missing,
    'Missing %'      : missing_pct,
    'Data Type'      : df.dtypes,
    'Unique Values'  : df.nunique(),
    'Min'            : df.min(),
    'Max'            : df.max(),
}).round(4)

print("Data Quality Report:")
print(quality_df.to_string())
print(f"\n✅ Total missing values: {missing.sum()}")
print(f"✅ All columns are numeric: {(df.dtypes == 'float64').all()}")


In [ ]:
# ── Duplicate detection ───────────────────────────────────────────────────
n_duplicates = df.duplicated().sum()
fraud_dups   = df[df['Class'] == 1].duplicated().sum()
legit_dups   = df[df['Class'] == 0].duplicated().sum()

print(f"Total duplicate rows : {n_duplicates:,}")
print(f"  ├── In fraud class : {fraud_dups:,}")
print(f"  └── In legit class : {legit_dups:,}")
print()

# Show sample duplicates
if n_duplicates > 0:
    dup_mask = df.duplicated(keep=False)
    print(f"Sample of duplicated rows (showing first 6):")
    display(df[dup_mask].sort_values('Amount').head(6))


In [ ]:
# ── Missing value heatmap ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor(C['dark'])

# Missing value bar (all zeros, shown as green checkmark visual)
ax = axes[0]
null_counts = df.isnull().sum().values
ax.barh(df.columns, null_counts,
        color=[C['green'] if v == 0 else C['red'] for v in null_counts],
        edgecolor='none', height=0.6)
ax.set_xlabel('Missing Values')
ax.set_title('Missing Value Check — All Columns', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(null_counts) + 1 if max(null_counts) > 0 else 5)
ax.text(0.5, 0.5, '✅ ZERO MISSING\nVALUES', transform=ax.transAxes,
        ha='center', va='center', fontsize=22, color=C['green'],
        fontweight='bold', alpha=0.4)
ax.grid(True, axis='x', alpha=0.3)

# Data types breakdown
ax = axes[1]
dtype_counts = df.dtypes.value_counts()
colors_dtype = [C['blue'], C['gold'], C['green'], C['red']]
wedges, texts, autotexts = ax.pie(
    dtype_counts.values,
    labels=[str(dt) for dt in dtype_counts.index],
    colors=colors_dtype[:len(dtype_counts)],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': C['dark'], 'linewidth': 2},
    textprops={'color': C['white']},
    pctdistance=0.75
)
ax.set_title('Data Type Distribution', fontsize=12, fontweight='bold')

fig.suptitle('DATA QUALITY ASSESSMENT', color=C['gold'],
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Class Imbalance Analysis <a id='4'></a>

> ⚠️ **Critical Finding:** This dataset has extreme class imbalance.  
> Only **0.17%** of transactions are fraudulent.  
> This means standard accuracy is **misleading** — a model predicting everything as 'Legitimate'  
> achieves 99.83% accuracy but catches **zero fraud**.


In [ ]:
# ── Class distribution table ──────────────────────────────────────────────
class_dist = df['Class'].value_counts().reset_index()
class_dist.columns = ['Class', 'Count']
class_dist['Label']   = class_dist['Class'].map({0: 'Legitimate', 1: 'Fraud'})
class_dist['Percent'] = (class_dist['Count'] / len(df) * 100).round(4)
class_dist['Ratio']   = class_dist['Count'].max() // class_dist['Count']

print("Class Distribution:")
print(class_dist.to_string(index=False))
print()
print(f"  Imbalance ratio : {(df['Class']==0).sum() // df['Class'].sum()} : 1  (Legit : Fraud)")
print(f"  For every 1 fraud there are ~{(df['Class']==0).sum() // df['Class'].sum()} legitimate transactions")


In [ ]:
# ── Class imbalance visualisation ─────────────────────────────────────────
fig = plt.figure(figsize=(18, 8))
fig.patch.set_facecolor(C['dark'])
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.4)

# KPI cards
kpis = [
    ('Total Transactions', f"{len(df):,}",          C['blue'],  '🏦'),
    ('Fraud Cases',        f"{df['Class'].sum():,}", C['red'],   '🚨'),
    ('Legitimate Cases',   f"{(df['Class']==0).sum():,}", C['green'], '✅'),
    ('Fraud Rate',         '0.1727%',                C['gold'],  '📊'),
]
for i, (title, val, color, icon) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(color + '1A')
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)
    ax.text(0.5, 0.68, val,   ha='center', va='center', fontsize=22,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.28, title, ha='center', va='center', fontsize=9.5,
            color=C['white'], alpha=0.85, transform=ax.transAxes)
    ax.text(0.85, 0.85, icon, ha='center', va='center', fontsize=18,
            transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])

# Pie chart
ax_pie = fig.add_subplot(gs[1, 0])
vals   = [(df['Class']==0).sum(), df['Class'].sum()]
labels = [f'Legitimate\n{vals[0]:,}', f'Fraud\n{vals[1]:,}']
wedges, texts, autotexts = ax_pie.pie(
    vals, labels=labels,
    colors=[LEGIT_COLOR, FRAUD_COLOR],
    autopct='%1.4f%%', startangle=90,
    wedgeprops={'edgecolor': C['dark'], 'linewidth': 2.5},
    textprops={'color': C['white'], 'fontsize': 9},
    pctdistance=0.78
)
for at in autotexts: at.set_fontsize(8)
ax_pie.set_title('Class Distribution', fontweight='bold')

# Bar chart
ax_bar = fig.add_subplot(gs[1, 1])
bars = ax_bar.bar(['Legitimate', 'Fraud'], vals,
                   color=[LEGIT_COLOR, FRAUD_COLOR],
                   edgecolor='none', width=0.5)
ax_bar.set_ylabel('Transaction Count')
ax_bar.set_title('Absolute Counts', fontweight='bold')
ax_bar.set_yscale('log')
for bar, val in zip(bars, vals):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                f'{val:,}', ha='center', color=C['white'], fontsize=9, fontweight='bold')
ax_bar.grid(True, axis='y', alpha=0.3)

# Imbalance ratio visual
ax_ratio = fig.add_subplot(gs[1, 2:])
ratio = (df['Class']==0).sum() // df['Class'].sum()
ax_ratio.set_facecolor(C['mid'])
ax_ratio.set_xlim(0, 1); ax_ratio.set_ylim(0, 1)
ax_ratio.set_xticks([]); ax_ratio.set_yticks([])
ax_ratio.text(0.5, 0.62, f'578 : 1', ha='center', va='center',
              fontsize=40, fontweight='bold', color=C['red'],
              transform=ax_ratio.transAxes)
ax_ratio.text(0.5, 0.35, 'Legitimate : Fraud Ratio', ha='center',
              color=C['white'], fontsize=12, transform=ax_ratio.transAxes)
ax_ratio.text(0.5, 0.18,
              'For every fraud transaction, there are 578 legitimate ones',
              ha='center', color=C['gray'], fontsize=9,
              transform=ax_ratio.transAxes)
ax_ratio.set_title('Imbalance Ratio', fontweight='bold')

fig.suptitle('CLASS IMBALANCE ANALYSIS — Why Accuracy Is Misleading Here',
             color=C['gold'], fontsize=14, fontweight='bold', y=1.01)
plt.savefig('reports/figures/eda_01_class_imbalance.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()
print("\n⚠️  A naive model predicting ALL = Legitimate achieves 99.83% accuracy")
print("   but catches ZERO fraud. Use ROC-AUC, PR-AUC, F1, Recall instead.")


## 5. Transaction Amount Analysis <a id='5'></a>

In [ ]:
# ── Amount statistics split by class ──────────────────────────────────────
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

amount_stats = pd.DataFrame({
    'Metric'     : ['Count','Mean','Median','Std Dev','Min','25th pct','75th pct','95th pct','Max'],
    'Legitimate' : [
        f"{len(legit):,}",
        f"${legit['Amount'].mean():.2f}",
        f"${legit['Amount'].median():.2f}",
        f"${legit['Amount'].std():.2f}",
        f"${legit['Amount'].min():.2f}",
        f"${legit['Amount'].quantile(0.25):.2f}",
        f"${legit['Amount'].quantile(0.75):.2f}",
        f"${legit['Amount'].quantile(0.95):.2f}",
        f"${legit['Amount'].max():.2f}",
    ],
    'Fraud' : [
        f"{len(fraud):,}",
        f"${fraud['Amount'].mean():.2f}",
        f"${fraud['Amount'].median():.2f}",
        f"${fraud['Amount'].std():.2f}",
        f"${fraud['Amount'].min():.2f}",
        f"${fraud['Amount'].quantile(0.25):.2f}",
        f"${fraud['Amount'].quantile(0.75):.2f}",
        f"${fraud['Amount'].quantile(0.95):.2f}",
        f"${fraud['Amount'].max():.2f}",
    ]
})
print(amount_stats.to_string(index=False))


In [ ]:
# ── Amount distributions ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor(C['dark'])

# 1) Raw amount histogram (log scale)
ax = axes[0, 0]
ax.hist(legit['Amount'].clip(0, 2000), bins=80, color=LEGIT_COLOR,
        alpha=0.65, label='Legitimate', density=True)
ax.hist(fraud['Amount'].clip(0, 2000), bins=40, color=FRAUD_COLOR,
        alpha=0.85, label='Fraud',      density=True)
ax.set_xlabel('Amount ($)  [capped at $2,000]')
ax.set_ylabel('Density')
ax.set_title('Amount Distribution\n(Raw, capped at $2K)', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, alpha=0.3)

# 2) Log-transformed amount
ax = axes[0, 1]
ax.hist(np.log1p(legit['Amount']), bins=80, color=LEGIT_COLOR,
        alpha=0.65, label='Legitimate', density=True)
ax.hist(np.log1p(fraud['Amount']), bins=40, color=FRAUD_COLOR,
        alpha=0.85, label='Fraud',      density=True)
ax.set_xlabel('log(1 + Amount)')
ax.set_ylabel('Density')
ax.set_title('Amount Distribution\n(Log-transformed)', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, alpha=0.3)

# 3) Box plot
ax = axes[0, 2]
bp_data = [legit['Amount'].clip(0, 500), fraud['Amount'].clip(0, 500)]
bp = ax.boxplot(bp_data, labels=['Legitimate', 'Fraud'],
                patch_artist=True, notch=True,
                medianprops={'color': C['gold'], 'linewidth': 2.5},
                whiskerprops={'color': C['white'], 'linewidth': 1.5},
                capprops={'color': C['white'], 'linewidth': 1.5},
                flierprops={'marker': 'o', 'markersize': 2,
                            'markerfacecolor': C['gray'], 'alpha': 0.4})
bp['boxes'][0].set_facecolor(LEGIT_COLOR + '66')
bp['boxes'][1].set_facecolor(FRAUD_COLOR + '66')
ax.set_ylabel('Amount ($)  [capped $500]')
ax.set_title('Box Plot Comparison\n(capped $500)', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# 4) Amount buckets
ax = axes[1, 0]
bins_edges = [0, 10, 50, 100, 500, 1000, 5000, np.inf]
bins_labels = ['<$10','$10-50','$50-100','$100-500','$500-1K','$1K-5K','>$5K']
legit['Amount_bin'] = pd.cut(legit['Amount'], bins=bins_edges, labels=bins_labels)
fraud['Amount_bin'] = pd.cut(fraud['Amount'], bins=bins_edges, labels=bins_labels)

legit_bin_pct = legit['Amount_bin'].value_counts(normalize=True).reindex(bins_labels)
fraud_bin_pct = fraud['Amount_bin'].value_counts(normalize=True).reindex(bins_labels)

x = np.arange(len(bins_labels)); w = 0.35
ax.bar(x - w/2, legit_bin_pct * 100, w, color=LEGIT_COLOR, alpha=0.85, label='Legitimate')
ax.bar(x + w/2, fraud_bin_pct * 100, w, color=FRAUD_COLOR, alpha=0.85, label='Fraud')
ax.set_xticks(x); ax.set_xticklabels(bins_labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('% of Transactions in Class')
ax.set_title('Amount Tier Distribution\n(% within each class)', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, axis='y', alpha=0.3)

# 5) CDF comparison
ax = axes[1, 1]
for data, color, label in [
    (legit['Amount'], LEGIT_COLOR, f'Legitimate (n={len(legit):,})'),
    (fraud['Amount'], FRAUD_COLOR, f'Fraud (n={len(fraud):,})')
]:
    sorted_amt = np.sort(data)
    cdf = np.arange(1, len(sorted_amt)+1) / len(sorted_amt)
    ax.plot(sorted_amt[sorted_amt <= 2000],
            cdf[sorted_amt <= 2000], lw=2.5, color=color, label=label)
ax.set_xlabel('Amount ($)  [up to $2,000]')
ax.set_ylabel('Cumulative Probability')
ax.set_title('Cumulative Distribution\n(Amount, capped $2K)', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, alpha=0.3)

# 6) Mean & median comparison
ax = axes[1, 2]
metrics_amt = {
    'Mean'   : [legit['Amount'].mean(),   fraud['Amount'].mean()],
    'Median' : [legit['Amount'].median(), fraud['Amount'].median()],
    'Std Dev': [legit['Amount'].std(),    fraud['Amount'].std()],
}
x2 = np.arange(len(metrics_amt)); w2 = 0.35
for i, (metric, vals) in enumerate(metrics_amt.items()):
    pass
bars_l = ax.bar(np.arange(3) - w2/2,
                [legit['Amount'].mean(), legit['Amount'].median(), legit['Amount'].std()],
                w2, color=LEGIT_COLOR, alpha=0.85, label='Legitimate')
bars_f = ax.bar(np.arange(3) + w2/2,
                [fraud['Amount'].mean(), fraud['Amount'].median(), fraud['Amount'].std()],
                w2, color=FRAUD_COLOR, alpha=0.85, label='Fraud')
ax.set_xticks(np.arange(3)); ax.set_xticklabels(['Mean', 'Median', 'Std Dev'])
ax.set_ylabel('Amount ($)')
ax.set_title('Statistical Comparison', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, axis='y', alpha=0.3)
for bar in bars_l:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'${bar.get_height():.0f}', ha='center', color=C['white'], fontsize=8)
for bar in bars_f:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'${bar.get_height():.0f}', ha='center', color=C['white'], fontsize=8)

fig.suptitle('TRANSACTION AMOUNT ANALYSIS',
             color=C['gold'], fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_02_amount_analysis.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()

# Statistical test
stat, p_val = mannwhitneyu(fraud['Amount'], legit['Amount'], alternative='two-sided')
print(f"\nMann-Whitney U test (Fraud vs Legit Amount):")
print(f"  U-statistic : {stat:,.0f}")
print(f"  p-value     : {p_val:.6f}")
print(f"  → {'Significant' if p_val < 0.05 else 'Not significant'} difference (α=0.05)")


## 6. Time-Based Analysis <a id='6'></a>

In [ ]:
# ── Engineer time features ────────────────────────────────────────────────
df['Hour']  = (df['Time'] / 3600).astype(int) % 24
df['Day']   = (df['Time'] / 86400).astype(int)
df['Hour_label'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print("Time feature engineering:")
print(f"  Time range : {df['Time'].min():.0f}s – {df['Time'].max():.0f}s")
print(f"  Hour range : {df['Hour'].min()} – {df['Hour'].max()}")
print(f"  Day range  : {df['Day'].min()} – {df['Day'].max()}  (~{df['Day'].max()+1} days of data)")
print()
print("Fraud by hour (top 5 riskiest):")
hourly = df.groupby('Hour').agg(
    total=('Class','count'),
    fraud_n=('Class','sum')
).assign(fraud_rate=lambda x: x['fraud_n']/x['total']*100)
print(hourly.sort_values('fraud_rate', ascending=False).head(5).to_string())


In [ ]:
# ── Time analysis plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor(C['dark'])

# 1) Transaction volume over time
ax = axes[0, 0]
ax.hist(legit['Time']/3600, bins=96, color=LEGIT_COLOR,
        alpha=0.65, label='Legitimate', density=True)
ax.hist(fraud['Time']/3600, bins=48, color=FRAUD_COLOR,
        alpha=0.85, label='Fraud',      density=True)
ax.set_xlabel('Hours from Start of Dataset')
ax.set_ylabel('Density')
ax.set_title('Transaction Volume Over Time', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.axvline(24, color=C['gold'], ls='--', alpha=0.6, label='Day boundary')
ax.axvline(48, color=C['gold'], ls='--', alpha=0.6)
ax.grid(True, alpha=0.3)

# 2) Hourly transaction count
ax = axes[0, 1]
hourly_legit = legit.groupby('Hour').size()
hourly_fraud = fraud.groupby('Hour').size()
ax.bar(hourly_legit.index, hourly_legit.values, color=LEGIT_COLOR,
       alpha=0.7, label='Legitimate', width=0.8)
ax2_twin = ax.twinx()
ax2_twin.plot(hourly_fraud.index, hourly_fraud.values,
              color=FRAUD_COLOR, lw=2.5, marker='o', ms=4, label='Fraud')
ax2_twin.tick_params(colors=C['white'])
ax2_twin.set_ylabel('Fraud Count', color=FRAUD_COLOR)
ax.set_xlabel('Hour of Day (0–23)')
ax.set_ylabel('Legitimate Count', color=LEGIT_COLOR)
ax.set_title('Transactions by Hour of Day', fontweight='bold')
ax.grid(True, alpha=0.3)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, labelcolor=C['white'], framealpha=0.3)

# 3) Fraud rate by hour
ax = axes[0, 2]
hourly_rate = hourly.reset_index()
bar_colors  = [C['red'] if r > hourly_rate['fraud_rate'].mean()
               else C['blue'] for r in hourly_rate['fraud_rate']]
ax.bar(hourly_rate['Hour'], hourly_rate['fraud_rate'],
       color=bar_colors, edgecolor='none', width=0.8)
ax.axhline(hourly_rate['fraud_rate'].mean(), color=C['gold'],
           ls='--', lw=1.8, label=f"Mean: {hourly_rate['fraud_rate'].mean():.3f}%")
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Fraud Rate (%)')
ax.set_title('Fraud Rate by Hour\n(Red = above average risk)', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, axis='y', alpha=0.3)

# 4) Heatmap — hour × day
ax = axes[1, 0]
pivot_fraud = df[df['Class']==1].groupby(['Day','Hour']).size().unstack(fill_value=0)
if pivot_fraud.shape[0] > 1 and pivot_fraud.shape[1] > 0:
    sns.heatmap(pivot_fraud, ax=ax, cmap='Reds',
                linewidths=0, cbar_kws={'label': 'Fraud Count'},
                xticklabels=4)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Day Number')
    ax.set_title('Fraud Heatmap — Hour × Day', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Insufficient data for heatmap',
            ha='center', transform=ax.transAxes, color=C['white'])

# 5) Daily fraud counts
ax = axes[1, 1]
daily_stats = df.groupby('Day').agg(
    total=('Class','count'), fraud_n=('Class','sum')
).reset_index()
ax.fill_between(daily_stats['Day'], daily_stats['total'],
                color=LEGIT_COLOR, alpha=0.4, label='Total')
ax2b = ax.twinx()
ax2b.plot(daily_stats['Day'], daily_stats['fraud_n'],
          color=FRAUD_COLOR, lw=2.5, marker='o', ms=3, label='Daily Fraud')
ax2b.tick_params(colors=C['white'])
ax2b.set_ylabel('Fraud Count', color=FRAUD_COLOR)
ax.set_xlabel('Day Number')
ax.set_ylabel('Total Transactions', color=LEGIT_COLOR)
ax.set_title('Daily Transaction Volume', fontweight='bold')
ax.grid(True, alpha=0.3)

# 6) Time KDE: fraud vs legit
ax = axes[1, 2]
legit_hours = legit['Hour']
fraud_hours = fraud['Hour']
ax.hist(legit_hours, bins=24, range=(0, 24), density=True,
        color=LEGIT_COLOR, alpha=0.5, label='Legitimate')
ax.hist(fraud_hours, bins=24, range=(0, 24), density=True,
        color=FRAUD_COLOR, alpha=0.75, label='Fraud')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Density')
ax.set_title('Hour Distribution\nFraud vs Legitimate', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.set_xticks(range(0, 24, 2))
ax.grid(True, alpha=0.3)

fig.suptitle('TIME-BASED ANALYSIS',
             color=C['gold'], fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_03_time_analysis.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 7. PCA Feature Analysis (V1–V28) <a id='7'></a>

> 📝 V1–V28 are the result of PCA transformation applied by the bank to protect  
> customer privacy. The original features are not available. Despite anonymization,  
> these features are still highly predictive of fraud.


In [ ]:
# ── Summary statistics for all V features ─────────────────────────────────
v_cols = [f'V{i}' for i in range(1, 29)]
print("PCA Feature Summary — Mean by Class:")
summary = df.groupby('Class')[v_cols].mean().T
summary.columns = ['Legitimate Mean', 'Fraud Mean']
summary['Difference'] = summary['Fraud Mean'] - summary['Legitimate Mean']
summary['Abs Diff']   = summary['Difference'].abs()
summary = summary.sort_values('Abs Diff', ascending=False)
print(summary.round(4).to_string())
print(f"\nTop 5 most discriminative features: {list(summary.index[:5])}")


In [ ]:
# ── PCA feature distributions overview ────────────────────────────────────
fig, axes = plt.subplots(4, 7, figsize=(22, 14))
fig.patch.set_facecolor(C['dark'])
axes_flat = axes.flatten()

for i, col in enumerate(v_cols):
    ax = axes_flat[i]
    ax.hist(legit[col], bins=50, color=LEGIT_COLOR, alpha=0.55,
            density=True, label='Legit')
    ax.hist(fraud[col], bins=30, color=FRAUD_COLOR, alpha=0.80,
            density=True, label='Fraud')
    ax.set_title(col, fontsize=9, fontweight='bold', color=C['gold'])
    ax.set_xticks([]); ax.set_yticks([])

# Hide unused subplots
for j in range(len(v_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=LEGIT_COLOR, alpha=0.65, label='Legitimate'),
    Patch(facecolor=FRAUD_COLOR, alpha=0.85, label='Fraud')
]
fig.legend(handles=legend_elements, loc='lower right',
           labelcolor=C['white'], framealpha=0.3, fontsize=11)

fig.suptitle('V1–V28 Feature Distributions (Fraud vs Legitimate)',
             color=C['gold'], fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.02, 1, 0.97])
plt.savefig('reports/figures/eda_04_pca_distributions.png', dpi=120,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── Mean difference bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(C['dark'])

diff = summary['Difference'].reindex(v_cols)
bar_colors = [C['red'] if v < 0 else C['blue'] for v in diff.values]
bars = ax.bar(diff.index, diff.values, color=bar_colors, edgecolor='none', width=0.7)
ax.axhline(0, color=C['white'], lw=1, alpha=0.5)
ax.axhline(diff.mean(), color=C['gold'], ls='--', lw=1.5,
           label=f'Mean diff: {diff.mean():.4f}')
ax.set_xlabel('PCA Feature')
ax.set_ylabel('Mean(Fraud) − Mean(Legitimate)')
ax.set_title('Mean Difference per Feature (Fraud − Legitimate)\n'
             'Blue = Fraud higher | Red = Legit higher',
             fontweight='bold', fontsize=12)
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('reports/figures/eda_05_feature_mean_diff.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── KS-test: which features separate classes best ─────────────────────────
ks_results = []
for col in v_cols + ['Amount']:
    stat, p = ks_2samp(fraud[col].dropna(), legit[col].dropna())
    ks_results.append({'Feature': col, 'KS Stat': stat, 'p-value': p,
                       'Significant': p < 0.05})

ks_df = pd.DataFrame(ks_results).sort_values('KS Stat', ascending=False)
print("Kolmogorov-Smirnov Test — Distribution Difference (Fraud vs Legit):")
print("(Higher KS Stat = more different distributions = better separator)")
print()
print(ks_df.head(15).to_string(index=False))
print()
print(f"Features significantly different (p<0.05): {ks_df['Significant'].sum()}/{len(ks_df)}")


## 8. Correlation Analysis <a id='8'></a>

In [ ]:
# ── Full correlation matrix ────────────────────────────────────────────────
corr_matrix = df[v_cols + ['Amount', 'Class']].corr()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor(C['dark'])

# Full heatmap
ax = axes[0]
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap_div = sns.diverging_palette(240, 10, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap_div, center=0,
            annot=False, ax=ax, linewidths=0.3, linecolor=C['dark'],
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (Lower Triangle)', fontweight='bold', fontsize=12)
ax.tick_params(labelsize=8)

# Correlation with target
ax = axes[1]
target_corr = corr_matrix['Class'].drop('Class').sort_values()
colors_corr = [C['red'] if v < 0 else C['green'] for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors_corr, edgecolor='none', height=0.7)
ax.axvline(0, color=C['white'], lw=1, alpha=0.5)
ax.set_xlabel('Pearson Correlation with Class (Fraud=1)')
ax.set_title('Feature Correlation with Target\n(Red = negative, Green = positive)',
             fontweight='bold', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)
for bar, val in zip(ax.patches, target_corr.values):
    ax.text(val + (0.003 if val >= 0 else -0.003),
            bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left' if val >= 0 else 'right',
            color=C['white'], fontsize=7.5)

fig.suptitle('CORRELATION ANALYSIS', color=C['gold'], fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_06_correlation.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()

print("Top 5 features most positively correlated with fraud:")
print(target_corr.tail(5).to_string())
print("\nTop 5 features most negatively correlated with fraud:")
print(target_corr.head(5).to_string())


In [ ]:
# ── High-correlation feature pairs ───────────────────────────────────────
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = upper_tri.stack().reset_index()
high_corr.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr = high_corr.assign(Abs_Corr=high_corr['Correlation'].abs())
high_corr = high_corr[high_corr['Feature_2'] != 'Class']
high_corr = high_corr.sort_values('Abs_Corr', ascending=False).head(10)

print("Top 10 most correlated feature pairs:")
print(high_corr[['Feature_1','Feature_2','Correlation']].to_string(index=False))
print()
print("Note: V1-V28 were designed to be uncorrelated by PCA.")
print("High correlations here might indicate meaningful interactions.")


## 9. Fraud vs Legitimate — Key Feature Deep Dive <a id='9'></a>

In [ ]:
# ── Top discriminating features — detailed violin/box plots ──────────────
# Use top features from KS test
top_features = ks_df.head(9)['Feature'].tolist()
if 'Amount' not in top_features:
    top_features[-1] = 'Amount'

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.patch.set_facecolor(C['dark'])

for ax, feat in zip(axes.flatten(), top_features):
    legit_vals = legit[feat].clip(
        legit[feat].quantile(0.01), legit[feat].quantile(0.99))
    fraud_vals = fraud[feat].clip(
        fraud[feat].quantile(0.01), fraud[feat].quantile(0.99))

    parts = ax.violinplot(
        [legit_vals, fraud_vals],
        positions=[0, 1], showmedians=True, showextrema=True
    )
    parts['cmedians'].set_color(C['gold'])
    parts['cmedians'].set_linewidth(2.5)
    for pc, color in zip(parts['bodies'], [LEGIT_COLOR, FRAUD_COLOR]):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    for key in ['cmins', 'cmaxes', 'cbars']:
        parts[key].set_color(C['white'])
        parts[key].set_linewidth(1.2)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Legitimate', 'Fraud'], fontsize=9)
    ax.set_title(feat, fontsize=11, fontweight='bold', color=C['gold'])
    ax.grid(True, axis='y', alpha=0.3)

    # Annotate medians
    ax.text(0, legit_vals.median(), f' {legit_vals.median():.2f}',
            va='center', color=LEGIT_COLOR, fontsize=8)
    ax.text(1, fraud_vals.median(), f' {fraud_vals.median():.2f}',
            va='center', color=FRAUD_COLOR, fontsize=8)

fig.suptitle('TOP DISCRIMINATING FEATURES — Violin Plots\n'
             '(Gold line = median | wider = more data at that value)',
             color=C['gold'], fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_07_violin_plots.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── KDE plots for top 6 features ─────────────────────────────────────────
top6 = ks_df.head(6)['Feature'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor(C['dark'])

for ax, feat in zip(axes.flatten(), top6):
    # Clip to 1st-99th percentile for visibility
    clip_lo = df[feat].quantile(0.005)
    clip_hi = df[feat].quantile(0.995)

    legit_vals = legit[feat].clip(clip_lo, clip_hi)
    fraud_vals = fraud[feat].clip(clip_lo, clip_hi)

    ax.hist(legit_vals, bins=60, density=True, color=LEGIT_COLOR,
            alpha=0.45, label='Legitimate')
    ax.hist(fraud_vals, bins=30, density=True, color=FRAUD_COLOR,
            alpha=0.75, label='Fraud')

    # Medians
    ax.axvline(legit_vals.median(), color=LEGIT_COLOR, lw=2, ls='--',
               alpha=0.9, label=f'Legit median: {legit_vals.median():.2f}')
    ax.axvline(fraud_vals.median(), color=FRAUD_COLOR, lw=2, ls='-.',
               alpha=0.9, label=f'Fraud median: {fraud_vals.median():.2f}')

    ax.set_title(f'{feat}  (KS={ks_df[ks_df["Feature"]==feat]["KS Stat"].values[0]:.4f})',
                 fontweight='bold', color=C['gold'])
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(labelcolor=C['white'], framealpha=0.25, fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('TOP 6 MOST DISCRIMINATIVE FEATURES — KDE Comparison',
             color=C['gold'], fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_08_kde_top_features.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 10. Outlier & Anomaly Analysis <a id='10'></a>

In [ ]:
# ── Z-score outlier detection ─────────────────────────────────────────────
from scipy.stats import zscore

z_scores  = df[v_cols + ['Amount']].apply(zscore).abs()
outlier_3 = (z_scores > 3).sum(axis=1)
outlier_5 = (z_scores > 5).sum(axis=1)

print("Outlier Analysis (Z-score thresholds):")
print(f"  Transactions with ≥1 feature |Z| > 3 : {(outlier_3 >= 1).sum():,} ({(outlier_3>=1).mean()*100:.2f}%)")
print(f"  Transactions with ≥1 feature |Z| > 5 : {(outlier_5 >= 1).sum():,} ({(outlier_5>=1).mean()*100:.2f}%)")
print()

# Compare outlier rate between classes
df['n_outlier_cols_3'] = outlier_3
df['n_outlier_cols_5'] = outlier_5
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print("Avg number of outlier features per transaction:")
print(f"  Legitimate: |Z|>3: {legit['n_outlier_cols_3'].mean():.3f} | |Z|>5: {legit['n_outlier_cols_5'].mean():.3f}")
print(f"  Fraud:      |Z|>3: {fraud['n_outlier_cols_3'].mean():.3f} | |Z|>5: {fraud['n_outlier_cols_5'].mean():.3f}")


In [ ]:
# ── Outlier visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(C['dark'])

# 1) Outlier count distribution
ax = axes[0]
ax.hist(legit['n_outlier_cols_3'], bins=20, color=LEGIT_COLOR,
        alpha=0.65, density=True, label='Legitimate')
ax.hist(fraud['n_outlier_cols_3'], bins=20, color=FRAUD_COLOR,
        alpha=0.85, density=True, label='Fraud')
ax.set_xlabel('Number of Features with |Z| > 3')
ax.set_ylabel('Density')
ax.set_title('Outlier Feature Count\nper Transaction', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, alpha=0.3)

# 2) Per-feature outlier rate
ax = axes[1]
outlier_pct_legit = (z_scores.loc[legit.index, v_cols] > 3).mean() * 100
outlier_pct_fraud = (z_scores.loc[fraud.index, v_cols] > 3).mean() * 100
x = np.arange(len(v_cols)); w = 0.38
ax.bar(x - w/2, outlier_pct_legit, w, color=LEGIT_COLOR, alpha=0.75, label='Legitimate')
ax.bar(x + w/2, outlier_pct_fraud, w, color=FRAUD_COLOR, alpha=0.85, label='Fraud')
ax.set_xticks(x[::4]); ax.set_xticklabels(v_cols[::4], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('% of Transactions with |Z| > 3')
ax.set_title('Outlier Rate per Feature\n(|Z|>3)', fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, axis='y', alpha=0.3)

# 3) Amount outliers
ax = axes[2]
q1 = df['Amount'].quantile(0.25); q3 = df['Amount'].quantile(0.75)
iqr = q3 - q1; upper = q3 + 1.5 * iqr
amt_outliers = df[df['Amount'] > upper]
print(f"Amount outliers (IQR method): {len(amt_outliers):,} transactions > ${upper:.2f}")
print(f"  Fraud in outliers: {amt_outliers['Class'].sum():,} ({amt_outliers['Class'].mean()*100:.2f}%)")

classes_outs = amt_outliers['Class'].value_counts()
ax.pie(classes_outs.values,
       labels=['Legitimate', 'Fraud'] if 0 in classes_outs.index else ['Fraud', 'Legitimate'],
       colors=[LEGIT_COLOR, FRAUD_COLOR],
       autopct='%1.2f%%', startangle=90,
       wedgeprops={'edgecolor': C['dark'], 'linewidth': 2},
       textprops={'color': C['white']})
ax.set_title(f'Class Split in High-Amount\nOutliers (> ${upper:.0f})', fontweight='bold')

fig.suptitle('OUTLIER & ANOMALY ANALYSIS', color=C['gold'], fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_09_outliers.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 11. Bivariate Analysis <a id='11'></a>

In [ ]:
# ── Amount vs top features scatter ────────────────────────────────────────
top4 = ks_df.head(4)['Feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor(C['dark'])

for ax, feat in zip(axes.flatten(), top4):
    # Sample for speed
    legit_s = legit.sample(min(3000, len(legit)), random_state=42)
    fraud_s  = fraud.sample(min(len(fraud), len(fraud)), random_state=42)

    ax.scatter(legit_s['Amount'].clip(0, 500), legit_s[feat],
               c=LEGIT_COLOR, alpha=0.25, s=8, label='Legitimate')
    ax.scatter(fraud_s['Amount'].clip(0, 500), fraud_s[feat],
               c=FRAUD_COLOR, alpha=0.85, s=20, marker='*', label='Fraud')

    ax.set_xlabel('Amount ($) [capped $500]')
    ax.set_ylabel(feat)
    ax.set_title(f'Amount vs {feat}', fontweight='bold', color=C['gold'])
    ax.legend(labelcolor=C['white'], framealpha=0.3, fontsize=9)
    ax.grid(True, alpha=0.2)

fig.suptitle('BIVARIATE ANALYSIS — Amount vs Top Features\n(★ = Fraud, dots = Legitimate)',
             color=C['gold'], fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_10_scatter_bivariate.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── Pairplot of top 5 discriminating features ─────────────────────────────
top5 = ks_df.head(4)['Feature'].tolist() + ['Amount']

# Sample for manageable pairplot
sample_legit = legit[top5 + ['Class']].sample(1000, random_state=42)
sample_fraud  = fraud[top5 + ['Class']]
sample_pair  = pd.concat([sample_legit, sample_fraud], ignore_index=True)

plt.style.use('dark_background')
pair_grid = sns.pairplot(
    sample_pair, hue='Class',
    palette={0: LEGIT_COLOR, 1: FRAUD_COLOR},
    plot_kws={'alpha': 0.4, 's': 12},
    diag_kind='hist',
    diag_kws={'alpha': 0.5, 'bins': 25},
    corner=True
)
pair_grid.figure.patch.set_facecolor(C['dark'])
pair_grid.figure.suptitle(
    'Pairplot — Top Discriminating Features (1K legit + all fraud sampled)',
    color=C['gold'], fontsize=12, fontweight='bold', y=1.01
)
# Fix legend
handles = pair_grid.legend.legend_handles
labels  = ['Legitimate', 'Fraud']
pair_grid.legend.remove()
pair_grid.figure.legend(handles, labels, loc='upper right',
                        labelcolor=C['white'], framealpha=0.3, fontsize=10)

plt.savefig('reports/figures/eda_11_pairplot.png', dpi=110,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── Hour × Amount heatmap ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(C['dark'])

bins_amt = [0, 10, 50, 200, 1000, np.inf]
lbls_amt = ['<$10','$10-50','$50-200','$200-1K','>$1K']
df['Amount_tier2'] = pd.cut(df['Amount'], bins=bins_amt, labels=lbls_amt)
df['Hour_grp']     = pd.cut(df['Hour'], bins=[0,6,12,18,24],
                             labels=['Night(0-6)','Morning(6-12)','Afternoon(12-18)','Evening(18-24)'],
                             right=False, include_lowest=True)

for ax, class_label, title, cmap_ in zip(
    axes,
    [0, 1],
    ['Legitimate Transactions — Hour×Amount Heatmap',
     'FRAUD Transactions — Hour×Amount Heatmap'],
    ['Blues', 'Reds']
):
    subset = df[df['Class'] == class_label]
    pivot  = subset.groupby(['Hour_grp', 'Amount_tier2'], observed=True).size().unstack(fill_value=0)
    sns.heatmap(pivot, ax=ax, cmap=cmap_, annot=True, fmt='d',
                linewidths=1, linecolor=C['dark'],
                annot_kws={'size': 10, 'color': C['white']},
                cbar_kws={'shrink': 0.8})
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Amount Tier')
    ax.set_ylabel('Time of Day')

fig.suptitle('TRANSACTION PATTERNS — Time of Day × Amount Tier',
             color=C['gold'], fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/eda_12_hour_amount_heatmap.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 12. Key Insights Summary <a id='12'></a>

In [ ]:
# ── Comprehensive EDA findings summary ────────────────────────────────────
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print("=" * 70)
print("  EDA INSIGHTS SUMMARY — CREDIT CARD FRAUD DETECTION")
print("=" * 70)

print("\n📊 DATASET OVERVIEW")
print(f"  Total transactions : {len(df):,}")
print(f"  Fraud transactions : {df['Class'].sum():,}  ({df['Class'].mean()*100:.4f}%)")
print(f"  Imbalance ratio    : {(df['Class']==0).sum() // df['Class'].sum()} : 1")
print(f"  Duplicates found   : {df.duplicated().sum():,}")
print(f"  Missing values     : 0  (clean dataset)")

print("\n💰 AMOUNT INSIGHTS")
print(f"  Avg fraud amount   : ${fraud['Amount'].mean():.2f}")
print(f"  Avg legit amount   : ${legit['Amount'].mean():.2f}")
print(f"  Median fraud amount: ${fraud['Amount'].median():.2f}")
print(f"  Median legit amount: ${legit['Amount'].median():.2f}")
print(f"  Largest fraud      : ${fraud['Amount'].max():,.2f}")
print(f"  Most fraud (<$500) : {(fraud['Amount']<500).mean()*100:.1f}% of all fraud")

print("\n⏰ TIME INSIGHTS")
peak_hour = hourly.sort_values('fraud_rate', ascending=False).index[0]
peak_rate = hourly.sort_values('fraud_rate', ascending=False)['fraud_rate'].iloc[0]
print(f"  Riskiest hour      : {peak_hour}:00 ({peak_rate:.3f}% fraud rate)")
print(f"  Dataset span       : ~{df['Day'].max()+1} days of transaction data")

print("\n🔍 FEATURE INSIGHTS")
top3_feats = ks_df.head(3)['Feature'].tolist()
print(f"  Top discriminating features (KS-test): {top3_feats}")
top_pos = target_corr.tail(3).index.tolist()
top_neg = target_corr.head(3).index.tolist()
print(f"  Most positively correlated with fraud  : {top_pos}")
print(f"  Most negatively correlated with fraud  : {top_neg}")

print("\n⚠️  MODELLING IMPLICATIONS")
print("  1. Dataset is HIGHLY IMBALANCED (0.17% fraud) →")
print("     Use SMOTE / class weights; evaluate with PR-AUC & F1, NOT accuracy")
print("  2. Apply StandardScaler to Amount & Amount_log features")
print("  3. V14, V4, V12, V17 are strongest fraud predictors")
print("  4. Time-of-day (Hour) adds signal — include as a feature")
print("  5. Duplicate rows should be handled before modelling")
print("  6. PCA features already uncorrelated — no need for further dimensionality reduction")

print("\n" + "=" * 70)
print("  EDA COMPLETE — All charts saved to reports/figures/")
print("=" * 70)


In [ ]:
# ── Final visual summary card ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor(C['dark'])
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.4)

# ROW 0: KPI cards
summary_kpis = [
    ('Transactions',    f"{len(df):,}",              C['blue']),
    ('Fraud Cases',     f"{df['Class'].sum():,}",     C['red']),
    ('Fraud Rate',      f"{df['Class'].mean()*100:.4f}%", C['gold']),
    ('Top Feature',     f"{ks_df.iloc[0]['Feature']}", C['teal']),
]
for i, (label, value, color) in enumerate(summary_kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(color + '22')
    for s in ax.spines.values(): s.set_edgecolor(color); s.set_linewidth(2)
    ax.text(0.5, 0.62, value, ha='center', va='center',
            fontsize=22, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center',
            fontsize=10, color=C['white'], alpha=0.8, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])

# ROW 1 LEFT: Amount box
ax_box = fig.add_subplot(gs[1, 0:2])
data_box = [legit['Amount'].clip(0,300).values, fraud['Amount'].clip(0,300).values]
bp = ax_box.boxplot(data_box, labels=['Legitimate', 'Fraud'],
                    patch_artist=True, notch=True,
                    medianprops={'color': C['gold'], 'linewidth': 2.5},
                    whiskerprops={'color': C['white'], 'linewidth': 1.5},
                    capprops={'color': C['white'], 'linewidth': 1.5},
                    flierprops={'marker': 'o', 'ms': 2, 'markerfacecolor': C['gray'], 'alpha': 0.3})
bp['boxes'][0].set_facecolor(LEGIT_COLOR + '55')
bp['boxes'][1].set_facecolor(FRAUD_COLOR + '55')
ax_box.set_title(f'Amount Distribution (capped $300)', fontweight='bold')
ax_box.set_ylabel('Amount ($)'); ax_box.grid(True, axis='y', alpha=0.3)

# ROW 1 RIGHT: top features correlation bar
ax_fc = fig.add_subplot(gs[1, 2:])
top_tc = target_corr.abs().sort_values(ascending=False).head(10)
bar_cols = [C['red'] if target_corr[f] < 0 else C['green'] for f in top_tc.index]
ax_fc.barh(top_tc.index[::-1], top_tc.values[::-1], color=bar_cols[::-1], edgecolor='none', height=0.7)
ax_fc.set_xlabel('|Pearson Correlation| with Fraud (Class)')
ax_fc.set_title('Top 10 Features by Correlation\nwith Fraud Target', fontweight='bold')
ax_fc.grid(True, axis='x', alpha=0.3)

fig.suptitle('EDA SUMMARY DASHBOARD — Credit Card Fraud Detection',
             color=C['gold'], fontsize=15, fontweight='bold', y=1.01)
plt.savefig('reports/figures/eda_00_summary_dashboard.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()
print("\n✅  All EDA charts saved to reports/figures/")
